# Mateli RNN-T Finnish ASR Baseline

This notebook evaluates the published `samuli/mateli` Finnish RNN-T checkpoint as a **baseline** (no team training) on the project's `test_cv`, `test_parl`, and `test_fleurs` splits. All test sets use the project's shared normalization and WER/CER implementation in `scripts/asr_eval.py`; each test set is reported separately.

Mateli is a Finnish-only, 59.6M-parameter streaming recognizer with an RNN-T decoder and greedy decoding. It expects mono 16 kHz float32 audio. The model card describes verbatim spoken Finnish output, so comparisons with cleaned written references can reflect transcript-style differences as well as recognition errors. Model weights and code are licensed CC BY-NC 4.0. Cite the model card: https://huggingface.co/samuli/mateli.

In [1]:
# Colab runtime setup. The shared project evaluator is fetched from the public repository.
%pip -q install datasets jiwer safetensors soundfile scipy huggingface_hub
!git clone -q https://github.com/zehraoz36/team-tokens-finnish-asr.git /content/team-tokens-finnish-asr

import os, sys, json, time, zipfile, platform
from pathlib import Path
import numpy as np
import torch
from datasets import Audio, load_dataset
from huggingface_hub import HfApi, snapshot_download

REPO_DIR = Path('/content/team-tokens-finnish-asr')
sys.path.insert(0, str(REPO_DIR / 'scripts'))
import asr_eval as E

MODEL_ID = 'samuli/mateli'
DATASET_ID = E.HF_DATASET
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
DATASET_REVISION = HfApi().dataset_info(DATASET_ID).sha
print('PyTorch:', torch.__version__)
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Mateli revision:', MODEL_REVISION)
print('Dataset revision:', DATASET_REVISION)

fatal: destination path '/content/team-tokens-finnish-asr' already exists and is not an empty directory.
PyTorch: 2.14.1+cu130
Device: Tesla T4
Mateli revision: 3dd0f094234d565dc923391eee15c4425feafdea
Dataset revision: 2dad7a2dd2b4e688e673b4b9c2526f8aac750ec7


## Load the pinned checkpoint

The model revision is resolved to a Hub commit SHA before downloading. `Mateli.from_pretrained` selects CUDA when available and otherwise uses CPU.

In [2]:
MODEL_DIR = Path('/content/mateli')
snapshot_download(repo_id=MODEL_ID, revision=MODEL_REVISION, local_dir=str(MODEL_DIR))
sys.path.insert(0, str(MODEL_DIR))
from mateli import Mateli

asr = Mateli.from_pretrained(str(MODEL_DIR))
print('Mateli loaded from pinned revision:', MODEL_REVISION)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Mateli loaded from pinned revision: 3dd0f094234d565dc923391eee15c4425feafdea


## Smoke check on development audio

This checks model loading, audio format, and transcription output on one development clip. It does not produce a test result.

In [3]:
dev = load_dataset(DATASET_ID, split='dev_cv', revision=DATASET_REVISION)
dev = dev.cast_column('audio', Audio(sampling_rate=E.SAMPLING_RATE))
sample = dev[0]['audio']['array'].astype(np.float32)
smoke_text = asr.transcribe(sample).text
assert isinstance(smoke_text, str) and smoke_text.strip(), 'Smoke transcription was empty.'
print('Smoke transcription:', smoke_text)

README.md:   0%|          | 0.00/988 [00:00<?, ?B/s]

data/train-00000-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  267MB            

data/train-00000-of-00002.parquet: downloading bytes:           |  0.00B            

data/train-00001-of-00002.parquet: reconstructing file:   0%|          |  0.00B /  270MB            

data/train-00001-of-00002.parquet: downloading bytes:           |  0.00B            

data/dev_cv-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  267MB            

... (16 outputs left out)

Generating test_cv split: 0 examples [00:00, ? examples/s]

Generating test_fleurs split: 0 examples [00:00, ? examples/s]

Generating test_parl split: 0 examples [00:00, ? examples/s]

Smoke transcription: punaisessa mekossa


## Evaluate the three held-out test sets

Each test split is loaded at the pinned dataset revision. The project evaluator applies the shared normalization exactly once to each raw hypothesis, computes corpus-level WER/CER, and saves per-clip predictions plus runtime metrics. Mateli uses greedy decoding; no language model or test-set tuning is applied.

In [4]:
def mateli_transcribe(batch):
    return [asr.transcribe(np.asarray(audio, dtype=np.float32)).text for audio in batch]

test_sets = {}
for split in E.TEST_SETS:
    ds = load_dataset(DATASET_ID, split=split, revision=DATASET_REVISION)
    test_sets[split] = ds.cast_column('audio', Audio(sampling_rate=E.SAMPLING_RATE))

SYSTEM = 'Mateli RNN-T baseline'
rows = E.evaluate_system(
    system=SYSTEM,
    checkpoint=MODEL_ID,
    transcribe_fn=mateli_transcribe,
    test_sets=test_sets,
    role='baseline',
    revision=MODEL_REVISION,
    settings='Finnish-only Mateli reference implementation; RNN-T greedy decoding; mono 16 kHz float32; no external language model; shared project text normalization and scoring',
    precision='float32',
    batch_size=1,
)
E.show_results()

Mateli RNN-T baseline test_cv:   0%|          | 0/1833 [00:00<?, ?it/s]

[Mateli RNN-T baseline] test_cv      WER= 15.02  CER=  3.12  RTF=0.0866


Mateli RNN-T baseline test_parl:   0%|          | 0/1913 [00:00<?, ?it/s]

[Mateli RNN-T baseline] test_parl    WER= 12.61  CER=  3.77  RTF=0.0867


Mateli RNN-T baseline test_fleurs:   0%|          | 0/916 [00:00<?, ?it/s]

[Mateli RNN-T baseline] test_fleurs  WER= 17.21  CER=  6.96  RTF=0.0851


wer                           cer  \
test_set                       test_cv test_parl test_fleurs test_cv   
role     system                                                        
baseline Mateli RNN-T baseline   15.02     12.61       17.21    3.12   

                                                      
test_set                       test_parl test_fleurs  
role     system                                       
baseline Mateli RNN-T baseline      3.77        6.96

## Package predictions and run metadata

The archive contains the Mateli score row and all three per-clip prediction files. The metadata records exact model and dataset revisions and runtime details.

In [5]:
from datetime import datetime, timezone

slug = E._slug(SYSTEM)
metadata = {
    'system': SYSTEM,
    'role': 'baseline (published checkpoint; no team fine-tuning)',
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'dataset_id': DATASET_ID,
    'dataset_revision': DATASET_REVISION,
    'test_splits': list(E.TEST_SETS),
    'sampling_rate_hz': E.SAMPLING_RATE,
    'decoding': 'RNN-T greedy decoding',
    'normalization_and_scoring': 'scripts/asr_eval.py from the cloned project repository',
    'hardware': E.hardware(),
    'torch_version': torch.__version__,
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'model_card': 'https://huggingface.co/samuli/mateli',
    'license': 'CC BY-NC 4.0',
}
metadata_path = Path('mateli_rnnt_run_metadata.json')
metadata_path.write_text(json.dumps(metadata, indent=2), encoding='utf-8')
zip_path = Path('mateli_rnnt_baseline_results.zip')
E.export_zip([SYSTEM], str(zip_path))
with zipfile.ZipFile(zip_path, 'a', compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(metadata_path, arcname='run_metadata.json')
print('Created:', zip_path.resolve())
print('Archive contents:', zipfile.ZipFile(zip_path).namelist())

mateli_rnnt_baseline_results.zip: 4 files
Created: /content/mateli_rnnt_baseline_results.zip
Archive contents: ['results/scores/mateli-rnn-t-baseline.csv', 'results/predictions/mateli-rnn-t-baseline__test_cv.csv', 'results/predictions/mateli-rnn-t-baseline__test_fleurs.csv', 'results/predictions/mateli-rnn-t-baseline__test_parl.csv', 'run_metadata.json']


In [6]:
from google.colab import files
files.download(str(zip_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>